# 04 · Name similarity features (exploration)

> **Data:** exploration on the **uncleaned** in-scope pairs (`pairs_scoped.parquet`), fixed on purpose (`C.PAIRS_SCOPED`, not `C.PAIRS_INPUT`). Kept as documentation of how the features were built and chosen; not re-run on the cleaned data, so the numbers in the conclusions stay valid. Final numbers come from notebook 07.

**Goal:** build candidate features from `name`, `alias`, `firstName`, `lastName` and check which ones are worth keeping.

| Group | Features |
| --- | --- |
| 1 · whole name | `n_sim_sort`, `n_sim_set`, `n_jaro_winkler`, `n_exact_name` |
| 2 · words | `n_token_coverage`, `n_phonetic_match`, `n_initials` |
| 3 · structure | `n_lastname_status`, `n_firstname_status`, `n_numbers`, `n_script` |
| 4 · companies only | `n_sim_core` (without legal form), `n_legal_form` |

Similarities are the **best value over all name × alias combinations** of the two sides, on normalised names (same normalisation as notebook 03).

**Status:** exploration only, nothing is written to `data/`. Uses `C.PAIRS_SCOPED` (the uncleaned pairs) and a **sample of 50,000 pairs**. Runtime about 3–5 minutes. Kernel: *Python (ds_sanctions)*.

## Feature glossary

All name features start with `n_`. "Best" means the highest value over all combinations of names and aliases of the two sides; "best pair of names" is the one combination with the highest `token_set_ratio`.

| Abbreviation | Stands for | Values |
| --- | --- | --- |
| `n_sim_sort` | name similarity, **token sort ratio**: words sorted alphabetically, then compared letter by letter (rapidfuzz) | 0–1, best |
| `n_sim_set` | name similarity, **token set ratio**: compares the shared words and the rest, so a name contained in a longer one scores high | 0–1, best |
| `n_jaro_winkler` | **Jaro-Winkler similarity** of the whole name strings; letter-level, rewards a matching start | 0–1, best |
| `n_exact_name` | an **identical full name** on both sides (normalised, word order ignored) | 0 / 1 |
| `n_token_coverage` | **word coverage**: share of words of the shorter name found in the longer one (identical or Jaro-Winkler ≥ 0.9) | 0–1, best pair |
| `n_phonetic_match` | **phonetic match**: share of words of the shorter name with the same Metaphone sound code in the other | 0–1, best pair |
| `n_initials` | **initials** consistent? ("J. Smith" vs "John Smith") | no_initials / compatible / conflict |
| `n_lastname_status` | comparison of the structured **lastName** fields | match / similar / conflict / missing |
| `n_firstname_status` | comparison of the structured **firstName** fields | match / similar / conflict / missing |
| `n_numbers` | **numbers inside the name** ("Services 2 Ltd" vs "Services 1 Ltd") | none / one_side / same / different |
| `n_script` | **writing system** of the two primary names (Latin, Cyrillic, …) | same / different |
| `n_sim_core` | companies only: `n_sim_sort` on the **core name without legal form** (LLC, GmbH, OOO …) | 0–1, best |
| `n_legal_form` | companies only: **legal forms** of the two sides | none / one_side / same / conflict |
| `n_freq_word_band` | **frequency of the rarest shared word**: in how many target entities it appears, as a band | -1 none shared, 0 = 0–2 (incl. not found), 1 = 3–10, 2 = 11–100, 3 = 101–1,000, 4 = >1,000 |
| `n_freq_name_band` | **frequency of the rarest shared full name**: how many target entities carry it, as a band | -1 none shared, 0 = 0–2 (incl. not found), 1 = 3–5, 2 = 6–20, 3 = >20 |

The two frequency features come from notebook 03 (`targets.simple`, entities collapsed by name + birth date; 0, 1 and 2 in one band because of the merge effect, see notebook 06). They are rebuilt below so this notebook runs on its own.

In [ ]:
import sys, re, time, unicodedata
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import jellyfish
from unidecode import unidecode
from rapidfuzz import fuzz, process
from rapidfuzz.distance import JaroWinkler
from sklearn.metrics import roc_auc_score
from screening import config as C
from screening.lists import values

SAMPLE_N = 50_000
MAX_NAMES = 30   # cap names + aliases per side (a few entities have hundreds)

cols = ["pair_id", "judgement", "left_schema", "right_schema"] + [
    f"{side}_{prop}" for side in ("left", "right") for prop in ("name", "alias", "firstName", "lastName")]
pairs = pd.read_parquet(C.PAIRS_SCOPED, columns=cols)
smp = pairs.sample(SAMPLE_N, random_state=C.RANDOM_SEED).reset_index(drop=True)
smp["y"] = (smp["judgement"] == "positive").astype(int)
smp["person"] = smp["left_schema"] == "Person"
print(f"input: {C.PAIRS_SCOPED.name}  ({len(pairs):,} pairs) -> sample {len(smp):,}")
print(f"persons {smp['person'].mean():.1%}, positive {smp['y'].mean():.1%}")

## 0 · Helpers

Normalisation as in notebook 03. One-letter words are kept here (needed for initials).

In [ ]:
_NON_ALNUM = re.compile(r"[^a-z0-9]+")

def normalize(s):
    if not s:
        return ""
    return " ".join(_NON_ALNUM.sub(" ", unidecode(s).lower()).split())

def side_names(row, side):
    """Normalised, de-duplicated names + aliases of one side (at most MAX_NAMES)."""
    out = []
    for n in values(row, side, ["name", "alias"]):
        k = normalize(n)
        if k and k not in out:
            out.append(k)
    return out[:MAX_NAMES]

def name_key(s):
    return " ".join(sorted(s.split()))

def script_of(s):
    for ch in s or "":
        if ch.isalpha():
            n = unicodedata.name(ch, "")
            return n.split(" ")[0] if n else "OTHER"
    return "NONE"

## 1–3 · Feature functions

- **Group 2** is computed on the *best pair of names* (highest `token_set_ratio`), not on all aliases at once, so extra aliases cannot inflate the coverage.
- **Coverage:** share of words of the *shorter* name that have a match in the other (identical, or Jaro-Winkler ≥ 0.9 for words longer than 2 letters).
- **Phonetic:** share of words of the shorter name whose Metaphone code appears in the other. English-centric, so expect less for Arabic or Slavic names.
- **Initials:** `no_initials` / `compatible` (every initial matches a word starting with that letter) / `conflict`.
- **First/last name fields:** `match` / `similar` (Jaro-Winkler ≥ 0.9 or matching initial) / `conflict` / `missing`.
- **Numbers in the name:** `none` / `one_side` / `same` / `different`, on the best pair of names.
- **Script:** writing system of the first letter of each side's primary name, `same` / `different`.

In [ ]:
def fuzzy_word_match(a, b):
    return a == b or (len(a) > 2 and len(b) > 2 and JaroWinkler.normalized_similarity(a, b) >= 0.9)

def shorter_longer(tl, tr):
    return (tl, tr) if len(tl) <= len(tr) else (tr, tl)

def coverage(tl, tr):
    short, long_ = shorter_longer(tl, tr)
    short = [t for t in short if len(t) > 1]
    if not short:
        return np.nan
    return sum(any(fuzzy_word_match(t, u) for u in long_) for t in short) / len(short)

def phonetic_share(tl, tr):
    short, long_ = shorter_longer(tl, tr)
    short = [t for t in short if len(t) > 1 and t.isalpha()]
    codes = {jellyfish.metaphone(u) for u in long_ if len(u) > 1 and u.isalpha()}
    if not short or not codes:
        return np.nan
    return sum(jellyfish.metaphone(t) in codes for t in short) / len(short)

def initials_status(tl, tr):
    il, ir = [t for t in tl if len(t) == 1], [t for t in tr if len(t) == 1]
    if not il and not ir:
        return "no_initials"
    ok = (all(any(u.startswith(t) for u in tr) for t in il)
          and all(any(u.startswith(t) for u in tl) for t in ir))
    return "compatible" if ok else "conflict"

def field_status(raw_l, raw_r):
    L = [x for x in (normalize(v) for v in raw_l) if x]
    R = [x for x in (normalize(v) for v in raw_r) if x]
    if not L or not R:
        return "missing"
    if set(L) & set(R):
        return "match"
    if max(JaroWinkler.normalized_similarity(a, b) for a in L for b in R) >= 0.9:
        return "similar"
    if any((len(a) == 1 or len(b) == 1) and a[0] == b[0] for a in L for b in R):
        return "similar"
    return "conflict"

def number_status(a, b):
    nl, nr = set(re.findall(r"\d+", a)), set(re.findall(r"\d+", b))
    if not nl and not nr:
        return "none"
    if not nl or not nr:
        return "one_side"
    return "same" if nl == nr else "different"

## 4 · Company features: legal forms

Legal forms are recognised on the normalised name (so Cyrillic forms appear transliterated, e.g. `obshchestvo s ogranichennoi otvetstvennost iu` = OOO = LLC). Short, ambiguous words such as "at" are left out on purpose. Extend the list if the examples below show misses.

In [ ]:
LEGAL_FORMS = {
    "llc":  ["llc", "l l c", "ooo", "limited liability company",
             "obshchestvo s ogranichennoi otvetstvennost iu", "obshchestvo s ogranichennoi otvetstvennostiu",
             "tovaristvo z obmezhenoiu vidpovidal nistiu", "tovarystvo z obmezhenoiu vidpovidalnistiu"],
    "ltd":  ["ltd", "limited", "pte ltd", "pty ltd"],
    "jsc":  ["jsc", "ao", "oao", "zao", "cjsc", "ojsc", "aktsionernoe obshchestvo", "joint stock company"],
    "pjsc": ["pjsc", "pao", "publichnoe aktsionernoe obshchestvo", "public joint stock company"],
    "plc":  ["plc", "public limited company"],
    "inc":  ["inc", "incorporated"],
    "corp": ["corp", "corporation"],
    "gmbh": ["gmbh", "gesellschaft mit beschrankter haftung"],
    "kg":   ["gmbh co kg", "kg"],
    "ag":   ["ag", "aktiengesellschaft"],
    "sa":   ["sa", "s a", "sociedad anonima", "societe anonyme"],
    "sarl": ["sarl", "s a r l"],
    "srl":  ["srl", "s r l"],
    "spa":  ["spa", "s p a"],
    "bv":   ["bv", "b v"],
    "nv":   ["nv", "n v"],
    "llp":  ["llp"],
    "fze":  ["fze", "fzco", "fzc", "fz llc"],
    "co":   ["co", "company"],
}
_forms = sorted(((v, k) for k, vs in LEGAL_FORMS.items() for v in vs), key=lambda x: -len(x[0]))
_FORM_RE = re.compile(r"\b(" + "|".join(re.escape(v) for v, _ in _forms) + r")\b")
_FORM_OF = dict(_forms)

def split_legal(name):
    """(core name without legal forms, set of canonical legal forms found)."""
    found = {_FORM_OF[m.group(1)] for m in _FORM_RE.finditer(name)}
    return " ".join(_FORM_RE.sub(" ", name).split()), found

def legal_status(fl, fr):
    if not fl and not fr:
        return "none"
    if not fl or not fr:
        return "one_side"
    return "same" if fl & fr else "conflict"

for ex in ["OOO \"Газпром добыча Ноябрьск\"", "Myanmar Yatai International Holding Group Co., Ltd.",
           "DG TECH SERVICES 2 LTD", "NAI EUROPE ENERGY GMBH & CO. KG", "Bank Dabrabyt Joint Stock Company"]:
    print(f"{ex!r:60} -> {split_legal(normalize(ex))}")

**Conclusion.**
- **Legal forms are found and removed correctly,** also in transliterated Cyrillic ("OOO" → llc) and in long English forms ("Joint Stock Company" → jsc).
- **"GmbH & Co. KG" is mapped to kg only,** so it counts as "same form" against another KG, but as a conflict against a plain GmbH.

**Next step:** none; the core name is what `n_sim_core` compares.

## Compute all features for the sample

In [ ]:
def pair_features(r):
    L, R = side_names(r, "left"), side_names(r, "right")
    f = {"pair_id": r.pair_id, "name_missing": int(not L or not R)}
    if not L or not R:
        return f
    m_sort = process.cdist(L, R, scorer=fuzz.token_sort_ratio)
    m_set = process.cdist(L, R, scorer=fuzz.token_set_ratio)
    m_jw = process.cdist(L, R, scorer=JaroWinkler.normalized_similarity)
    f["n_sim_sort"] = m_sort.max() / 100
    f["n_sim_set"] = m_set.max() / 100
    f["n_jaro_winkler"] = float(m_jw.max())
    f["n_exact_name"] = int(bool({name_key(x) for x in L} & {name_key(x) for x in R}))

    i, j = np.unravel_index(m_set.argmax(), m_set.shape)     # best pair of names
    tl, tr = L[i].split(), R[j].split()
    f["n_token_coverage"] = coverage(tl, tr)
    f["n_phonetic_match"] = phonetic_share(tl, tr)
    f["n_initials"] = initials_status(tl, tr)

    f["n_lastname_status"] = field_status(values(r, "left", ["lastName"]), values(r, "right", ["lastName"]))
    f["n_firstname_status"] = field_status(values(r, "left", ["firstName"]), values(r, "right", ["firstName"]))
    f["n_numbers"] = number_status(L[i], R[j])
    raw_l, raw_r = values(r, "left", ["name"]), values(r, "right", ["name"])
    f["n_script"] = ("same" if script_of(raw_l[0] if raw_l else "") == script_of(raw_r[0] if raw_r else "")
                     else "different")

    if r.person:
        f["n_sim_core"], f["n_legal_form"] = np.nan, "n/a"
    else:
        sl, sr = [split_legal(x) for x in L], [split_legal(x) for x in R]
        cl, cr = [c for c, _ in sl if c], [c for c, _ in sr if c]
        f["n_sim_core"] = (process.cdist(cl, cr, scorer=fuzz.token_sort_ratio).max() / 100
                           if cl and cr else np.nan)
        f["n_legal_form"] = legal_status(set().union(*(s for _, s in sl)), set().union(*(s for _, s in sr)))
    return f

t0 = time.time()
feats = pd.DataFrame([pair_features(r) for r in smp.itertuples(index=False)])
feats = feats.merge(smp[["pair_id", "y", "judgement", "person"]], on="pair_id")
print(f"{len(feats):,} pairs in {time.time() - t0:.0f}s; names missing on a side: {feats['name_missing'].mean():.1%}")
feats.head()

## Evaluation

For every feature, separately for persons and companies/organisations:
- **numeric features:** AUC on its own (0.5 = useless, 1.0 = separates perfectly) and the share of negative pairs per similarity band;
- **categorical features:** share of negative pairs per category.

A good feature: the share of negatives falls clearly as similarity rises, or one category is clearly "safe" or clearly "suspicious".

### Table: AUC per feature

**What it shows:** how well each numeric feature *on its own* separates same-entity pairs (positive) from different-entity pairs (negative), for persons and for companies separately.

**How to read it:** AUC is the probability that a randomly chosen positive pair gets a higher value than a randomly chosen negative pair. 0.5 = no better than chance, 1.0 = perfect separation. `pairs` is the number of pairs where the feature could be computed (`n_sim_core` only exists for companies). Sorted from best to worst.

In [ ]:
NUM = ["n_sim_sort", "n_sim_set", "n_jaro_winkler", "n_exact_name", "n_token_coverage",
       "n_phonetic_match", "n_sim_core"]
CAT = ["n_initials", "n_lastname_status", "n_firstname_status", "n_numbers", "n_script", "n_legal_form"]
GROUPS = {"PERSON": feats[feats["person"]], "COMPANY / ORG": feats[~feats["person"]]}

def auc_table(df):
    rows = []
    for c in NUM:
        s = df[[c, "y"]].dropna()
        if len(s) >= 100 and s["y"].nunique() == 2:
            rows.append((c, len(s), round(roc_auc_score(s["y"], s[c]), 3)))
    return pd.DataFrame(rows, columns=["feature", "pairs", "AUC"]).sort_values("AUC", ascending=False)

for title, df in GROUPS.items():
    print(title); display(auc_table(df))

**Conclusion.**
- **Every similarity score is strong on its own** (AUC 0.93–0.98).
- **Persons:** `n_sim_sort` is best (0.962).
- **Companies:** the similarity without legal form (`n_sim_core`, 0.983) beats every full-name score, so stripping the legal form helps.

**Next step:** AUC is close to its ceiling for all of them; compare them at a fixed recall in a model (section 5) instead.

### Tables: share of negatives and number of pairs per similarity band

**What they show:** the pairs are grouped by the value of each feature (e.g. all pairs with `n_sim_sort` between 0.8 and 0.9). The first table gives the share of *negative* pairs in each band, the second how many pairs fall into it.

**How to read them:** for a useful feature the share of negatives falls from close to 1 at low similarity to close to 0 at 1.0, ideally gradually, so the band tells you something. Always check the second table: a band with very few pairs (e.g. 1 or 30) gives an unreliable share. A feature where almost all pairs sit in "<0.5" or "1.0" is nearly binary and carries little detail.

In [ ]:
bins = [-0.001, 0.5, 0.7, 0.8, 0.9, 0.95, 0.999, 1.0]
labels = ["<0.5", "0.5-0.7", "0.7-0.8", "0.8-0.9", "0.9-0.95", "0.95-<1", "1.0"]
band_cols = ["n_sim_sort", "n_sim_set", "n_jaro_winkler", "n_token_coverage", "n_phonetic_match", "n_sim_core"]

for title, df in GROUPS.items():
    neg = {}
    cnt = {}
    for c in band_cols:
        b = pd.cut(df[c], bins, labels=labels)
        neg[c] = df.groupby(b, observed=False)["y"].apply(lambda s: round(1 - s.mean(), 3) if len(s) else np.nan)
        cnt[c] = b.value_counts().reindex(labels)
    print(f"{title} · share of negative pairs per band"); display(pd.DataFrame(neg))
    print(f"{title} · number of pairs per band"); display(pd.DataFrame(cnt))

**Conclusion.**
- **The share of negatives falls steeply only at the very top of each score:** for persons, `n_sim_sort` 0.9–0.95 is 8.3% negative, 1.0 is 2.8%, but 0.8–0.9 is already 54%.
- **Most pairs sit at 1.0** (22,323 of 32,280 persons), so the scores cannot separate the many pairs with identical names.
- **Companies** need even higher similarity (0.95–<1: 32% negative).

**Next step:** add features that distinguish pairs with the same name (frequency, first/last name fields, legal form).

### Tables: share of negatives per category

**What they show:** for each categorical feature, the number of negative and positive pairs per category and the share of negatives (`neg_share`).

**How to read them:** compare `neg_share` across categories. A category far above the overall rate (persons about 29%, companies about 54% in this sample) is a warning sign ("conflict"); one far below is reassuring ("match"). Small categories (a handful of pairs) are not reliable. A feature whose categories all have about the same `neg_share` adds nothing.

In [ ]:
for title, df in GROUPS.items():
    print(title)
    for c in CAT:
        if df[c].nunique() < 2:
            continue
        t = pd.crosstab(df[c], df["judgement"])
        t["neg_share"] = (t.get("negative", 0) / t.sum(axis=1)).round(3)
        display(t.rename_axis(c))

**Conclusion.**
- **Last-name and first-name fields are strong where present:** a last-name conflict is 95.9% negative, a match 5.7%.
- **Legal forms:** a conflict is 99.2% negative for companies.
- **Numbers in company names:** different numbers are 98.7% negative, the same numbers 7.7%.
- **Initials and script are weak** (28–40% negative for persons).

**Next step:** keep all of them; the model decides how much the weak ones add (section 14).

### Table: correlation between the numeric features

**What it shows:** Spearman rank correlation between each pair of numeric features, for persons and companies.

**How to read it:** 1.0 means two features put the pairs in exactly the same order; above about 0.9 they carry nearly the same information and one of them can probably go without loss. Low values mean a feature measures something different and may add information. High correlation alone does not prove a feature is useless (a model can still use small differences), but together with a lower AUC it is a strong hint.

In [ ]:
for title, df in GROUPS.items():
    print(title)
    display(df[[c for c in NUM if df[c].notna().any()]].corr(method="spearman").round(2))

**Conclusion.**
- **The similarity scores are highly correlated** (Spearman 0.82–0.99): token coverage, phonetic match and token-set similarity are almost the same measure (0.97–0.99).
- **Single weights of linear models will therefore be hard to interpret** (see section 13).

**Next step:** keep them for the tree models, which cope with correlated inputs, and check later whether some are redundant (section 14).

## The hard cases: identical names

**What it shows:** only the pairs that share an identical full name (`n_exact_name = 1`, after normalisation and ignoring word order), split into persons (`True`) and companies (`False`). Whole-name similarity is 1.0 for all of them by construction, so the tables only show the structural features.

**How to read it:** `neg_share` per category within this subset. If a category raises `neg_share` clearly above the subset's overall rate, that feature can still catch false alarms among identical names. If all categories stay close to the overall rate, name features have reached their limit here, and the rest must come from birth date, country etc. (Vova's record features) and from name frequency.

In [ ]:
hard = feats[feats["n_exact_name"] == 1]
print(f"pairs with an identical full name: {len(hard):,}, of which negative: {(hard['y'] == 0).sum():,} ({(hard['y'] == 0).mean():.1%})")
for c in CAT:
    t = pd.crosstab([hard["person"], hard[c]], hard["judgement"])
    if "negative" not in t:
        continue
    t["neg_share"] = (t["negative"] / t.sum(axis=1)).round(3)
    display(t.rename_axis(["person", c]))

**Conclusion.**
- **Only 2.4% of the 29,261 pairs with an identical full name are negative,** but those 703 are the core of the false-alarm problem.
- **Among identical names the name features hardly help:** a last-name conflict raises the share to 11.7%, everything else stays at 1–7%.

**Next step:** the frequency of the shared name (notebook 03) and record features (notebook 05) are needed for these pairs.

## Findings so far (2026-11-13, uncleaned pairs, sample of 50,000)

Preliminary, to be confirmed on the cleaned pairs.

- **All similarity scores separate well, but measure nearly the same thing.** AUC 0.93–0.96 for persons, 0.94–0.98 for companies; correlations mostly 0.9–0.99 (`n_token_coverage` vs `n_phonetic_match` 0.99, `n_sim_set` vs `n_token_coverage` 0.98, `n_sim_sort` vs `n_exact_name` 0.98).
- **`n_sim_sort` is the best whole-name score for persons (AUC 0.962), `n_sim_core` for companies (0.983, vs 0.965 with the legal form left in).** Removing the legal form helps.
- **`n_token_coverage` and `n_phonetic_match` are almost binary** (nearly all pairs at 1.0 or below 0.5) and add little beyond `n_sim_sort`.
- **Last/first name fields are strong for persons:** `n_lastname_status = conflict` 96% negative (5,620 pairs), `match` 6%; `n_firstname_status = conflict` 85%. About 37% of person pairs lack the fields (`missing`).
- **For companies, legal form and numbers are strong but rare:** `n_legal_form = conflict` 99% negative (1,353 pairs), `one_side` 86%; `n_numbers = different` or `one_side` 97–99% negative (470 pairs), `same` 8%.
- **Weak:** `n_script` (persons 31% vs 28%), `n_initials` for persons (conflict 40% vs 29%).
- **Identical names:** 29,261 sample pairs share an identical full name; 2.4% of them are negative (703). Within this group the structural features help only a little (last name conflict 12% vs match 1.5%): the rest must come from record features and name frequency.

Candidate set: `n_sim_sort` (persons) / `n_sim_core` (companies), `n_lastname_status`, `n_firstname_status`, `n_legal_form`, `n_numbers`, plus the two frequency features from notebook 03.

## 5 · First model: what do the features buy at a fixed recall?

AUC per feature is close to its ceiling. What matters for the project is the number of **false alarms at a fixed, very high recall** (95–99.5%). So we train a quick model with different feature sets and compare exactly that.

- **Model:** scikit-learn `HistGradientBoostingClassifier` (gradient-boosted trees, like LightGBM, already installed; handles missing values).
- **Split:** 70% train / 30% test **by connected components**: pairs that share an entity stay on the same side, so the model cannot memorise entities.
- **Frequency features:** the tables from notebook 03 are rebuilt here (about 1–2 minutes).
- Still the uncleaned sample of 50,000 pairs: the numbers are for comparing feature sets, not final results.

In [ ]:
# rebuild the frequency tables (same rules as notebook 03)
from collections import Counter
TARGETS_CSV = C.DATA_RAW / "targets.simple-20261111.csv"
FREQ_GROUPS = {"person": ("Person",), "org": ("Company", "Organization", "LegalEntity")}

t0 = time.time()
tg = pd.read_csv(TARGETS_CSV, usecols=["id", "schema", "name", "aliases", "birth_date"],
                 dtype=str, keep_default_na=False)
tg = tg[tg["schema"].isin(FREQ_GROUPS["person"] + FREQ_GROUPS["org"])]
key = tg["name"].map(lambda n: name_key(normalize(n)))
tg["identity"] = np.where(tg["birth_date"].str.strip() != "", key + " | dob " + tg["birth_date"], "id " + tg["id"])
tg = tg.drop_duplicates(subset=["schema", "identity"])     # same name + same birth date = one person

word_freq, name_freq = {}, {}
for group, schemas in FREQ_GROUPS.items():
    words, names = Counter(), Counter()
    for name, aliases in zip(*(tg.loc[tg["schema"].isin(schemas), c] for c in ("name", "aliases"))):
        variants = [normalize(v) for v in [name] + (aliases.split(";") if aliases else [])]
        variants = [v for v in variants if v]
        words.update({w for v in variants for w in v.split() if len(w) > 1})
        names.update({name_key(v) for v in variants})
    word_freq[group], name_freq[group] = words, names
print(f"frequency tables rebuilt in {time.time() - t0:.0f}s")

# 0 (name not in targets.simple) is merged with 1-2: both are almost always positive (merge effect, notebook 06)
WORD_BANDS = [(0, 2), (3, 10), (11, 100), (101, 1000), (1001, 10**9)]   # -> 0..4
NAME_BANDS = [(0, 2), (3, 5), (6, 20), (21, 10**9)]                       # -> 0..3
def band(n, bands):
    return next(i for i, (lo, hi) in enumerate(bands) if lo <= n <= hi)

def freq_features(r):
    L, R = side_names(r, "left"), side_names(r, "right")
    g = "person" if r.person else "org"
    sw = {w for n in L for w in n.split() if len(w) > 1} & {w for n in R for w in n.split() if len(w) > 1}
    sn = {name_key(n) for n in L} & {name_key(n) for n in R}
    return {"pair_id": r.pair_id,
            "n_freq_word_band": band(min(word_freq[g].get(w, 0) for w in sw), WORD_BANDS) if sw else -1,
            "n_freq_name_band": band(min(name_freq[g].get(k, 0) for k in sn), NAME_BANDS) if sn else -1}

ff = pd.DataFrame([freq_features(r) for r in smp.itertuples(index=False)])
data = feats.merge(ff, on="pair_id")
print(data[["n_freq_word_band", "n_freq_name_band"]].apply(pd.Series.value_counts).sort_index())

In [ ]:
# train/test split by connected components of the pair graph
ids = pd.read_parquet(C.PAIRS_SCOPED, columns=["pair_id", "left_id", "right_id"])
data = data.merge(ids, on="pair_id")

parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
for a, b_ in zip(data["left_id"], data["right_id"]):
    parent[find(a)] = find(b_)
data["component"] = data["left_id"].map(find)

from sklearn.model_selection import GroupShuffleSplit
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=C.RANDOM_SEED)
                           .split(data, groups=data["component"]))
train, test = data.iloc[train_idx], data.iloc[test_idx]
print(f"train {len(train):,} pairs, test {len(test):,} pairs ({test['y'].sum():,} positive, {(test['y'] == 0).sum():,} negative)")
print(f"components: {data['component'].nunique():,}; entities in both train and test: "
      f"{len(set(train[['left_id', 'right_id']].values.ravel()) & set(test[['left_id', 'right_id']].values.ravel()))}")

### Feature sets compared

| Set | Features |
| --- | --- |
| **A · similarity only** | `n_sim_sort` (a name-only score, like a legacy screening engine) |
| **B · all 15** | every name feature in the glossary |
| **C · reduced 8** | `n_sim_sort`, `n_sim_core`, `n_lastname_status`, `n_firstname_status`, `n_legal_form`, `n_numbers`, `n_freq_word_band`, `n_freq_name_band` |
| **D · C without frequency** | C minus the two `n_freq_*` |
| **E · C without first/last name fields** | C minus `n_lastname_status`, `n_firstname_status` |
| **F · C without company features** | C minus `n_sim_core`, `n_legal_form`, `n_numbers` |

Every set also gets `is_person` (person or company pair), so the model can treat the two differently. Categorical features are one-hot encoded.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

ALL = NUM + CAT + ["n_freq_word_band", "n_freq_name_band"]
REDUCED = ["n_sim_sort", "n_sim_core", "n_lastname_status", "n_firstname_status", "n_legal_form",
           "n_numbers", "n_freq_word_band", "n_freq_name_band"]
SETS = {
    "A · similarity only": ["n_sim_sort"],
    "B · all 15": ALL,
    "C · reduced 8": REDUCED,
    "D · C without frequency": [c for c in REDUCED if not c.startswith("n_freq")],
    "E · C without first/last name": [c for c in REDUCED if c not in ("n_lastname_status", "n_firstname_status")],
    "F · C without company features": [c for c in REDUCED if c not in ("n_sim_core", "n_legal_form", "n_numbers")],
}
RECALLS = [0.95, 0.98, 0.99, 0.995]

def design(df, cols):
    X = df[cols].copy()
    X["is_person"] = df["person"].astype(int)
    return pd.get_dummies(X, columns=[c for c in cols if c in CAT], dtype=float)

def false_alarms_at(y, score, recall):
    """Lowest threshold that still reaches the recall; returns (threshold, false alarms)."""
    order = np.argsort(-score, kind="stable")
    tp, fp = np.cumsum(y[order]), np.cumsum(1 - y[order])
    k = np.searchsorted(tp, np.ceil(recall * y.sum()))
    return score[order][k], int(fp[k])

results, scores = [], {}
for name, cols in SETS.items():
    Xtr, Xte = design(train, cols), design(test, cols)
    Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
    model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, random_state=C.RANDOM_SEED)
    model.fit(Xtr, train["y"])
    s = model.predict_proba(Xte)[:, 1]
    scores[name] = s
    row = {"feature set": name, "features": len(cols), "AUC": round(roc_auc_score(test["y"], s), 4)}
    for r in RECALLS:
        row[f"false alarms @ {r:.1%}"] = false_alarms_at(test["y"].to_numpy(), s, r)[1]
    results.append(row)
res_table = pd.DataFrame(results).set_index("feature set")
res_table

**Conclusion.**
- **All 15 features (set B) cut the false alarms at 99% recall from 5,501 (similarity only) to 649,** an 88% reduction.
- **The reduced set C** (8 features) gets 1,121.
- **Leaving out frequency (D) or company features (F)** roughly doubles the false alarms against C; first/last name (E) matters less.

**Next step:** these numbers come from one split; repeat with several splits (sections 8–11).

### Table: false alarms at fixed recall

**What it shows:** for each feature set, how many of the negative test pairs (different entities) would raise an alert if the threshold is set low enough to catch 95%, 98%, 99% or 99.5% of the positive test pairs.

**How to read it:** fewer false alarms at the same recall = better feature set. Compare along a column. B vs C shows whether dropping 7 features costs anything; C vs D/E/F shows what each feature group is worth. A vs C shows what everything beyond a plain name score buys. Look mainly at 99%: that is the operating point of the project. The numbers are counts on this test set, not production alert rates (the data is post-blocking).

## 6 · Where the remaining errors are (set C, threshold at 99% recall)

Two lists from the test set:
1. **False alarms with the highest score:** different entities the model is most sure are the same. These drive the false-alarm count.
2. **Hardest true matches (lowest score):** same entities the model almost missed. They force the threshold down to reach 99% recall.

For each pair: the primary names and the key features.

In [ ]:
s = scores["C · reduced 8"]
thr, fa = false_alarms_at(test["y"].to_numpy(), s, 0.99)
view = test.assign(score=s).merge(smp[["pair_id", "left_name", "right_name"]], on="pair_id")
view["left"] = view["left_name"].map(lambda v: v[0] if len(v) else "")
view["right"] = view["right_name"].map(lambda v: v[0] if len(v) else "")
show = ["left", "right", "score", "n_sim_sort", "n_lastname_status", "n_firstname_status",
        "n_freq_word_band", "n_freq_name_band", "n_legal_form"]
print(f"threshold for 99% recall: {thr:.4f}; false alarms: {fa:,} of {(view['y'] == 0).sum():,} negatives")
print("\n1) false alarms with the highest score")
display(view[(view["y"] == 0) & (view["score"] >= thr)].sort_values("score", ascending=False)[show].head(25))
print("2) hardest true matches (lowest score)")
display(view[view["y"] == 1].sort_values("score")[show].head(25))

**Conclusion.**
- **The highest-scoring false alarms are near-identical names** of different entities: Russian LLCs with the same short name, Korean names with the same syllables, transliterations.
- **The hardest true matches show two fixable gaps:** titles in person names ("Maulavi", "Général de brigade") and legal forms that were not recognised (Romanian SRL, Turkish ŞTİ, Estonian OÜ). Others differ by script (Japanese katakana) or by missing parts of the name.

**Next step:** strip titles and extend the legal-form list (version 2, section 7).

## 7 · Closing the gaps found in section 6 (version 2 of the features)

The hardest true matches in section 6 showed two fixable gaps:

1. **Legal forms missing from the list**, so they stay in the "core" name and lower `n_sim_core` (similarity without legal form): e.g. Romanian *Societatea cu Răspundere Limitată* = S.R.L., Turkish *Ltd. Şti.*, Estonian *OÜ*, Russian *Открытое акционерное общество* (OAO written out).
2. **Titles and ranks in person names** lower every similarity: e.g. *Général de brigade Burhan QADOUR*, *Maulavi Hamdullah*.

Version 2 extends the legal-form list and removes titles from person names (only if a name remains). Everything else is unchanged. The features of section 1–4 stay as version 1 for comparison.

Not fixed: Japanese katakana transliterations (*ロマン・レパ* = Roman Lepa) are converted poorly by `unidecode`; that is left as a known limitation.

In [ ]:
TITLES = ["general de brigade", "general de division", "general de corps d armee", "brigadier general",
          "major general", "lieutenant general", "lieutenant colonel", "general", "brigadier", "colonel", "col",
          "major", "captain", "capt", "lieutenant", "lt", "admiral", "commander",
          "sheikh", "shaykh", "sheik", "maulavi", "mawlawi", "maulvi", "mullah", "mulla", "mufti", "haji", "hajji",
          "dr", "prof", "professor", "mr", "mrs", "ms", "miss", "sir", "his excellency", "eng", "engineer"]
_TITLE_RE = re.compile(r"\b(" + "|".join(re.escape(t) for t in sorted(TITLES, key=len, reverse=True)) + r")\b")

def strip_titles(name):
    out = " ".join(_TITLE_RE.sub(" ", name).split())
    return out or name

LEGAL_FORMS_V2 = {k: list(v) for k, v in LEGAL_FORMS.items()}
LEGAL_FORMS_V2["srl"] += ["societatea cu raspundere limitata", "societate cu raspundere limitata",
                          "sociedad de responsabilidad limitada", "societa a responsabilita limitata"]
LEGAL_FORMS_V2["sa"] += ["societate pe actiuni", "sa de cv", "s a de c v", "anonim sirketi", "a s"]
LEGAL_FORMS_V2["ltd"] += ["ltd sti", "limited sirketi", "pvt ltd", "private limited", "sdn bhd", "bhd"]
LEGAL_FORMS_V2["jsc"] += ["otkrytoe aktsionernoe obshchestvo", "zakrytoe aktsionernoe obshchestvo",
                          "aktsionerne tovaristvo", "aktsionerne tovarystvo",
                          "privatne aktsionerne tovaristvo", "pryvatne aktsionerne tovarystvo"]
LEGAL_FORMS_V2.update({
    "ou": ["ou"], "sro": ["s r o", "sro", "spol s r o"], "spzoo": ["sp z o o", "spolka z ograniczona odpowiedzialnoscia"],
    "kft": ["kft"], "doo": ["d o o", "doo"], "ltda": ["ltda"], "as": ["as", "asa"], "ab": ["ab"],
    "oy": ["oy", "oyj"], "aps": ["aps"],
})
_forms2 = sorted(((v, k) for k, vs in LEGAL_FORMS_V2.items() for v in vs), key=lambda x: -len(x[0]))
_FORM_RE2 = re.compile(r"\b(" + "|".join(re.escape(v) for v, _ in _forms2) + r")\b")
_FORM_OF2 = dict(_forms2)

def split_legal_v2(name):
    found = {_FORM_OF2[m.group(1)] for m in _FORM_RE2.finditer(name)}
    return " ".join(_FORM_RE2.sub(" ", name).split()), found

side_names_v1, split_legal_v1 = side_names, split_legal
def side_names_v2(row, side):
    names = side_names_v1(row, side)
    return list(dict.fromkeys(strip_titles(n) for n in names)) if row.person else names

for ex in ["Général de brigade Burhan QADOUR", "Maulavi Hamdullah", "Dr. He Wei"]:
    print(f"{ex!r:40} -> {strip_titles(normalize(ex))!r}")
for ex in ["Societatea cu Răspundere Limitată JET4U", "FIRST OIL JV LTD. ŞTİ.", "Kihonzi buzhaga OÜ",
           'ОТКРЫТОЕ АКЦИОНЕРНОЕ ОБЩЕСТВО "30 СУДОРЕМОНТНЫЙ ЗАВОД"']:
    print(f"{ex!r:60} -> {split_legal_v2(normalize(ex))}")

**Conclusion.** Titles are now stripped ("Maulavi Hamdullah" → "hamdullah") and the new legal forms are recognised (SRL, ŞTİ, OÜ, Russian JSC).

**Next step:** compute version 2 and compare.

In [ ]:
# recompute all name features with version 2 (functions of sections 1-5 use these names at call time)
side_names, split_legal = side_names_v2, split_legal_v2
try:
    t0 = time.time()
    feats_v2 = pd.DataFrame([pair_features(r) for r in smp.itertuples(index=False)])
    ff_v2 = pd.DataFrame([freq_features(r) for r in smp.itertuples(index=False)])
finally:
    side_names, split_legal = side_names_v1, split_legal_v1
data_v2 = (feats_v2.merge(ff_v2, on="pair_id")
           .merge(data[["pair_id", "y", "judgement", "person", "component"]], on="pair_id"))
print(f"version 2 computed in {time.time() - t0:.0f}s")

# which pairs changed most?
cmp_v = data[["pair_id", "n_sim_sort", "n_sim_core"]].merge(
    data_v2[["pair_id", "n_sim_sort", "n_sim_core"]], on="pair_id", suffixes=("_v1", "_v2"))
for c in ["n_sim_sort", "n_sim_core"]:
    d = cmp_v[f"{c}_v2"] - cmp_v[f"{c}_v1"]
    print(f"{c}: higher in v2 for {(d > 0.01).sum():,} pairs, lower for {(d < -0.01).sum():,}")

**Conclusion.** Version 2 changes few values: `n_sim_sort` for about 200 pairs, `n_sim_core` for about 1,600 (mostly lower, because more legal forms are now removed before comparing).

**Next step:** compare versions 1 and 2 in the model with cross-validation (section 8).

## 8 · Securing the numbers: 5-fold cross-validation

One split is not enough at 99–99.5% recall, where only a few dozen true matches decide the threshold. Here: **5 folds by connected component** (every pair is in the test part exactly once; pairs sharing an entity stay together).

Because folds differ in size, false alarms are reported as a **share of the negative test pairs** (false-alarm rate, in %), as mean ± standard deviation over the 5 folds.

Compared:
- version 1 vs version 2 for sets **B** (all 15) and **C** (reduced 8): what do the gap fixes buy?
- all six sets **A–F** on version 2: are the earlier conclusions stable?

In [ ]:
from sklearn.model_selection import GroupKFold

def cv_eval(df, sets, version, k=5):
    rows = []
    for fold, (tr, te) in enumerate(GroupKFold(n_splits=k).split(df, groups=df["component"])):
        train_, test_ = df.iloc[tr], df.iloc[te]
        y = test_["y"].to_numpy()
        for name, cols in sets.items():
            Xtr, Xte = design(train_, cols), design(test_, cols)
            Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
            m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, random_state=C.RANDOM_SEED)
            s = m.fit(Xtr, train_["y"]).predict_proba(Xte)[:, 1]
            row = {"version": version, "set": name, "fold": fold, "test_pairs": len(y), "negatives": int((y == 0).sum())}
            for r in RECALLS:
                row[f"{r:.1%}"] = 100 * false_alarms_at(y, s, r)[1] / (y == 0).sum()
            rows.append(row)
    return pd.DataFrame(rows)

t0 = time.time()
cv = pd.concat([
    cv_eval(data, {k: SETS[k] for k in ["B · all 15", "C · reduced 8"]}, "v1"),
    cv_eval(data_v2, SETS, "v2"),
])
print(f"{cv['fold'].nunique()} folds, {len(cv)} model fits in {time.time() - t0:.0f}s")
print("test pairs per fold:", cv.drop_duplicates(["version", "fold"]).query("version == 'v2'")["test_pairs"].tolist())

### Table: false-alarm rate at fixed recall, mean ± sd over 5 folds

**What it shows:** for each version and feature set, the share of negative test pairs (in %) that would raise an alert at the threshold that catches 95 / 98 / 99 / 99.5% of the true matches. Mean over the 5 folds, ± standard deviation.

**How to read it:** lower is better. A difference between two rows is only convincing if it is clearly larger than the ± values. v1 vs v2 (same set) shows the effect of the gap fixes; the v2 rows A–F show what each feature group is worth.

In [ ]:
pct = [f"{r:.1%}" for r in RECALLS]
summary = cv.groupby(["version", "set"])[pct].agg(["mean", "std"])
table = pd.DataFrame({p_: summary[(p_, "mean")].round(2).astype(str) + " ± " + summary[(p_, "std")].round(2).astype(str)
                      for p_ in pct})
table.columns = [f"false-alarm % @ {p_} recall" for p_ in pct]
table

**Conclusion.**
- **Version 2 is equal to version 1** within the standard deviation (set B at 99% recall: 10.1% vs 10.2%).
- **The ± values are very large** (about ±4 pp), because the folds differ strongly in size and content.

**Next step:** find out why the folds differ (section 9).

## 9 · The giant component

The folds in section 8 differ a lot in size (one fold about 20,800 pairs, the others about 7,300). The reason: many pairs are linked through shared entities into one **giant connected component**, which a group split must keep in one piece. This section looks at what it is.

**Tables below:**
1. *Component sizes:* the largest components in the sample and in the full data.
2. *Giant component vs the rest:* how the pairs inside it differ (share positive, persons, name similarity).
3. *Hub entities:* the entities with the most pairs inside the giant component; they hold it together.
4. *Source lists:* which datasets its records come from.

In [ ]:
# 1) component sizes in the sample and in the full data
sizes = data["component"].value_counts()
giant = sizes.index[0]
print(f"sample: {len(sizes):,} components; largest: {sizes.iloc[0]:,} pairs ({sizes.iloc[0] / len(data):.1%} of the sample); "
      f"next: {sizes.iloc[1:6].tolist()}")

full = pd.read_parquet(C.PAIRS_SCOPED, columns=["pair_id", "left_id", "right_id"])
parent_full = {}
def find_full(x):
    parent_full.setdefault(x, x)
    while parent_full[x] != x:
        parent_full[x] = parent_full[parent_full[x]]
        x = parent_full[x]
    return x
for a, b_ in zip(full["left_id"], full["right_id"]):
    parent_full[find_full(a)] = find_full(b_)
full_sizes = full["left_id"].map(find_full).value_counts()
print(f"full data: {len(full_sizes):,} components; largest: {full_sizes.iloc[0]:,} pairs ({full_sizes.iloc[0] / len(full):.1%} of all pairs); "
      f"next: {full_sizes.iloc[1:6].tolist()}")

In [ ]:
# 2) giant component vs the rest of the sample
data["in_giant"] = data["component"] == giant
def describe(df):
    return pd.Series({"pairs": len(df), "share positive %": 100 * df["y"].mean(),
                      "share persons %": 100 * df["person"].mean(),
                      "median n_sim_sort": df["n_sim_sort"].median(),
                      "identical full name %": 100 * df["n_exact_name"].mean()})
pd.DataFrame({"giant component": describe(data[data["in_giant"]]),
              "rest": describe(data[~data["in_giant"]])}).round(2)

**Conclusion.**
- **The giant component is a different population:** 30% positive (rest 86%), half companies, median similarity only 0.67, identical full name in 29% of pairs (rest 81%).
- **Whichever fold gets it is much easier** for false alarms, which explains the large ± values.

**Next step:** compare folds pairwise (section 10) and look for a split that spreads the giant component (section 11).

In [ ]:
# 3) hub entities and 4) source lists inside the giant component
extra = pd.read_parquet(C.PAIRS_SCOPED, columns=["pair_id", "left_id", "right_id", "left_schema", "right_schema",
                                                 "left_name", "right_name", "left_datasets", "right_datasets"])
gx = extra[extra["pair_id"].isin(data.loc[data["in_giant"], "pair_id"])]
ents = pd.concat([
    gx[["left_id", "left_schema", "left_name"]].set_axis(["id", "schema", "name"], axis=1),
    gx[["right_id", "right_schema", "right_name"]].set_axis(["id", "schema", "name"], axis=1)])
ents["name"] = ents["name"].map(lambda v: v[0] if len(v) else "")
hubs = (ents.groupby("id").agg(pairs=("id", "size"), schema=("schema", "first"), name=("name", "first"))
            .sort_values("pairs", ascending=False))
print(f"entities in the giant component: {len(hubs):,}; pairs per entity: median {hubs['pairs'].median():.0f}, max {hubs['pairs'].max():,}")
display(hubs.head(15))

ds = pd.concat([gx["left_datasets"], gx["right_datasets"]]).explode().value_counts()
print("source lists of the records in the giant component (top 15):")
display(ds.head(15).rename("records").to_frame())

**Conclusion.**
- **The giant component is held together by heavily sanctioned organisations** that were compared with many candidates (up to 39 pairs each: Hayat Tahrir al-Sham, Iran Air, Alfa-Bank, Uralvagonzavod …).
- **Their records come from the big sanctions lists** (US, EU, UK, France, Switzerland).

**Next step:** a split by connected component cannot spread it; split by entity instead (section 11).

## 10 · Paired comparison per fold

The ± values in section 8 mostly reflect how different the folds are (one of them holds the giant component), not how uncertain the comparison between feature sets is. Comparing two sets **within the same fold** removes that: if set B beats set C in every fold, the difference is real even when the ± values overlap.

**Tables below:**
1. *False-alarm rate per fold:* one row per fold, one column per feature set (version 2, at 99% recall), plus the fold size. The fold with the giant component is marked.
2. *Difference to set C per fold:* each set minus C (in percentage points) at 99% and 99.5% recall. Negative = fewer false alarms than C. The last columns count in how many of the 5 folds the set is better than C, and give the mean difference without the giant fold.

In [ ]:
v2cv = cv[cv["version"] == "v2"].copy()
giant_fold = v2cv.loc[v2cv["test_pairs"].idxmax(), "fold"]

per_fold = v2cv.pivot_table(index="fold", columns="set", values="99.0%").round(2)
per_fold.insert(0, "test pairs", v2cv.drop_duplicates("fold").set_index("fold")["test_pairs"])
per_fold.index = [f"{f} (giant)" if f == giant_fold else str(f) for f in per_fold.index]
print("false-alarm % at 99% recall, per fold")
display(per_fold)

rows = []
for rec in ["99.0%", "99.5%"]:
    wide = v2cv.pivot_table(index="fold", columns="set", values=rec)
    for s in wide.columns:
        if s.startswith("C "):
            continue
        d = wide[s] - wide["C · reduced 8"]
        rows.append({"recall": rec, "set": s,
                     **{f"fold {f}{' (giant)' if f == giant_fold else ''}": round(v, 2) for f, v in d.items()},
                     "folds better than C": int((d < 0).sum()),
                     "mean diff without giant fold": round(d.drop(giant_fold).mean(), 2)})
print("difference to set C in percentage points (negative = fewer false alarms than C)")
pd.DataFrame(rows).set_index(["recall", "set"])

**Conclusion.**
- **Per fold, the ranking is stable:** set B beats C in 5 of 5 folds, and leaving out frequency (D) or company features (F) makes it worse in every fold.
- **The giant fold has 3.5% false alarms, the others 11–14%.**

**Next step:** use a split that does not depend on the giant component (section 11).

## 11 · Option A: split by entity instead of by component (all pairs)

The split by connected component (sections 5–10) has to keep the giant component (57.5% of all pairs) in one piece, so folds cannot be balanced. Here instead:

- Each **entity** is put into the test side with probability 30%, otherwise into the train side.
- A pair is a **test pair** if both its entities are on the test side, a **train pair** if both are on the train side. Pairs with one entity on each side are dropped for that split. No entity is in both train and test.
- Repeated **5 times** with different random seeds; the giant component is spread over train and test every time.
- Run on **all pairs** (not the 50,000 sample), so that the test sets are large enough for 99–99.5% recall. Computing the features for all pairs takes about 8 minutes.

Uses version 2 of the features (section 7).

In [ ]:
allp = pairs.copy()
allp["y"] = (allp["judgement"] == "positive").astype(int)
allp["person"] = allp["left_schema"] == "Person"

side_names, split_legal = side_names_v2, split_legal_v2
try:
    t0 = time.time()
    F_all = pd.DataFrame([pair_features(r) for r in allp.itertuples(index=False)])
    FF_all = pd.DataFrame([freq_features(r) for r in allp.itertuples(index=False)])
finally:
    side_names, split_legal = side_names_v1, split_legal_v1

full_data = (F_all.merge(FF_all, on="pair_id")
             .merge(allp[["pair_id", "y", "judgement", "person"]], on="pair_id")
             .merge(full[["pair_id", "left_id", "right_id"]], on="pair_id"))
full_data["component"] = full_data["left_id"].map(find_full)
full_data["in_giant"] = full_data["component"] == full_sizes.index[0]
print(f"features for {len(full_data):,} pairs in {time.time() - t0:.0f}s; "
      f"in the giant component: {full_data['in_giant'].mean():.1%}")

In [ ]:
def entity_split(df, q, seed):
    rng = np.random.default_rng(seed)
    ents = pd.unique(pd.concat([df["left_id"], df["right_id"]]))
    is_test = pd.Series(rng.random(len(ents)) < q, index=ents)
    lt, rt = df["left_id"].map(is_test), df["right_id"].map(is_test)
    return df[~lt & ~rt], df[lt & rt]

Q, REPS = 0.3, 5
rows, sub_rows = [], []
t0 = time.time()
for rep in range(REPS):
    tr, te = entity_split(full_data, Q, C.RANDOM_SEED + rep)
    y = te["y"].to_numpy()
    giant_mask = te["in_giant"].to_numpy()
    for name, cols in SETS.items():
        Xtr, Xte = design(tr, cols), design(te, cols)
        Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
        m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, random_state=C.RANDOM_SEED)
        s = m.fit(Xtr, tr["y"]).predict_proba(Xte)[:, 1]
        row = {"rep": rep, "set": name, "train_pairs": len(tr), "test_pairs": len(te), "negatives": int((y == 0).sum())}
        for r in RECALLS:
            row[f"{r:.1%}"] = 100 * false_alarms_at(y, s, r)[1] / (y == 0).sum()
        rows.append(row)
        if name in ("B · all 15", "C · reduced 8"):
            for grp, mask in [("giant component", giant_mask), ("rest", ~giant_mask)]:
                yy, ss = y[mask], s[mask]
                sr = {"rep": rep, "set": name, "part": grp, "test_pairs": len(yy), "share positive %": 100 * yy.mean()}
                for r in RECALLS:
                    sr[f"{r:.1%}"] = 100 * false_alarms_at(yy, ss, r)[1] / (yy == 0).sum()
                sub_rows.append(sr)
    print(f"rep {rep}: train {len(tr):,}, test {len(te):,} pairs "
          f"(kept {(len(tr) + len(te)) / len(full_data):.0%} of all pairs; test in giant: {giant_mask.mean():.0%})  "
          f"[{time.time() - t0:.0f}s]")
ecv, esub = pd.DataFrame(rows), pd.DataFrame(sub_rows)

### Tables: entity split, false-alarm rate and paired comparison

**First table:** share of negative test pairs (in %) that would raise an alert at the threshold for 95 / 98 / 99 / 99.5% recall; mean ± standard deviation over the 5 repetitions. Lower is better.

**Second table:** for each repetition, the difference of each set to set C (reduced 8) in percentage points; negative = fewer false alarms than C. The last column counts in how many of the 5 repetitions the set beats C. If a set wins in 5 of 5, the difference is reliable even when the ± values overlap.

In [ ]:
pct = [f"{r:.1%}" for r in RECALLS]
summ = ecv.groupby("set")[pct].agg(["mean", "std"])
display(pd.DataFrame({f"false-alarm % @ {p_}": summ[(p_, "mean")].round(2).astype(str) + " ± " + summ[(p_, "std")].round(2).astype(str)
                      for p_ in pct}))

drows = []
for rec in ["99.0%", "99.5%"]:
    wide = ecv.pivot_table(index="rep", columns="set", values=rec)
    for s_ in wide.columns:
        if s_.startswith("C "):
            continue
        d = wide[s_] - wide["C · reduced 8"]
        drows.append({"recall": rec, "set": s_, **{f"rep {k}": round(v, 2) for k, v in d.items()},
                      "reps better than C": int((d < 0).sum())})
pd.DataFrame(drows).set_index(["recall", "set"])

**Conclusion.**
- **With the entity split on all pairs,** set B reaches 2.92% at 99% recall (± 0.19), similarity only 19.68%.
- **Every feature group helps in 5 of 5 repetitions:** without frequency 6.04%, without first/last name 5.00%, without company features 7.13%.

**Next step:** check giant component and rest separately (next table).

### Table: giant component vs rest (sets B and C)

**What it shows:** the same models, but the false-alarm rate is computed separately for the test pairs inside the giant component and for the rest, each with its own threshold for the given recall. Mean over the 5 repetitions.

**How to read it:** if the gap between B and C is similar in both parts, the features help everywhere; if one part is much harder, the overall number mostly reflects that part.

In [ ]:
esub.groupby(["part", "set"])[["test_pairs", "share positive %"] + pct].mean().round(2)

**Conclusion.** Inside the giant component set B has 1.6% false alarms, in the rest 20.9%. The hard cases sit outside the giant component, where few negatives exist but most of them look like matches.

**Next step:** cross-check on the rest alone (section 12).

## 12 · Option B: without the giant component (cross-check)

All pairs **outside** the giant component, 5-fold split by connected component (works here, the remaining components are small). Not representative on its own: these pairs are 86% positive and mostly near-identical names. It shows whether the ranking of the feature sets also holds there.

In [ ]:
rest_all = full_data[~full_data["in_giant"]]
rest_sets = {k: SETS[k] for k in ["A · similarity only", "B · all 15", "C · reduced 8", "D · C without frequency"]}
t0 = time.time()
cv_rest = cv_eval(rest_all, rest_sets, "rest only")
print(f"{len(rest_all):,} pairs, {cv_rest['fold'].nunique()} folds, {time.time() - t0:.0f}s; "
      f"test pairs per fold: {cv_rest.drop_duplicates('fold')['test_pairs'].tolist()}")
s_ = cv_rest.groupby("set")[pct].agg(["mean", "std"])
pd.DataFrame({f"false-alarm % @ {p_}": s_[(p_, "mean")].round(2).astype(str) + " ± " + s_[(p_, "std")].round(2).astype(str)
              for p_ in pct})

**Conclusion.** On the rest alone, set B gets 20.5% at 99% recall, similarity only 41.7%: the features still halve the false alarms where the task is hardest, and the ranking of the sets is the same.

**Next step:** none; the result holds with and without the giant component.

## 13 · Lasso and Ridge: a linear view on the 15 features

Logistic regression with a penalty, on all 15 features (set B), standardised so the weights are comparable:

- **Ridge (L2 penalty):** shrinks all weights but keeps every feature. A weight is the change in log-odds of "same entity" per standard deviation of the feature.
- **Lasso (L1 penalty):** sets weights to exactly 0. Lowering the penalty step by step (the *Lasso path*) shows in which **order** features enter the model: early = carries information the others do not have.

**Limits (why this complements, not replaces, the ablation in section 11):** correlated features share or split their weight (Ridge spreads it, Lasso keeps one more or less at random); a linear model sees no interactions (e.g. legal form only matters for companies); and the weights describe the fit over all pairs, not the false alarms at 99% recall.

Setup: train/test from the entity split of section 11 (first repetition); a random 100,000 of the training pairs for speed. Missing values (e.g. `n_sim_core` for persons) are set to 0 with an extra indicator column "(missing)". Categorical features become one column per category; for comparisons per feature the absolute weights of its columns are summed.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

tr0, te0 = entity_split(full_data, Q, C.RANDOM_SEED)
tr_s = tr0.sample(min(100_000, len(tr0)), random_state=C.RANDOM_SEED)

def linear_design(df, cols):
    X = design(df, cols)
    for c in list(X.columns):
        if X[c].isna().any():
            X[c + " (missing)"] = X[c].isna().astype(float)
            X[c] = X[c].fillna(0)
    return X

Xtr = linear_design(tr_s, ALL)
lin_cols = Xtr.columns
Xte = linear_design(te0, ALL).reindex(columns=lin_cols, fill_value=0).fillna(0)
scaler = StandardScaler().fit(Xtr)
Ztr, Zte = scaler.transform(Xtr), scaler.transform(Xte)

def base_of(col):
    for f in sorted(ALL + ["is_person"], key=len, reverse=True):
        if col.startswith(f):
            return f
    return col
feature_of = pd.Series([base_of(c) for c in lin_cols], index=lin_cols)
print(f"{len(tr_s):,} training pairs, {len(te0):,} test pairs, {len(lin_cols)} columns from {feature_of.nunique()} features")

### Tables: Ridge weights

**First table:** every column with its Ridge weight (per standard deviation) and the odds ratio `exp(weight)`: above 1 = makes "same entity" more likely, below 1 = less likely. Sorted by absolute weight.

**Second table:** per feature, the sum of the absolute weights of its columns, as a rough importance.

**How to read them:** large absolute weight = the model relies on it. But remember the limits above: the five whole-name similarities (`n_sim_sort`, `n_sim_set`, `n_jaro_winkler`, `n_token_coverage`, `n_phonetic_match`) are 0.9–0.99 correlated, so each one's weight understates the group; weights can even get opposite signs that cancel out.

In [ ]:
ridge = LogisticRegression(penalty="l2", C=1.0, solver="liblinear", max_iter=2000).fit(Ztr, tr_s["y"])
w = pd.Series(ridge.coef_[0], index=lin_cols)
display(pd.DataFrame({"feature": feature_of, "weight (per SD)": w.round(3), "odds ratio": np.exp(w).round(3)})
        .reindex(w.abs().sort_values(ascending=False).index))
w.abs().groupby(feature_of).sum().sort_values(ascending=False).round(3).to_frame("sum of |weights|")

**Conclusion.**
- **The largest Ridge weights go to exact name, core similarity (companies) and the full-name frequency** (negative: common names lower the odds).
- **Correlated similarity scores get opposite signs** (Jaro-Winkler negative), so single weights cannot be read as importance.

**Next step:** use the Lasso path and the model comparison instead (next tables).

### Lasso path

**What it shows:** Lasso fitted with the penalty strength going from very strong (left, small C) to weak (right, large C). The plot shows, per feature, the sum of its absolute weights; the table gives the C at which each feature first gets a non-zero weight and its rank.

**How to read it:** features entering first carry the most information on their own; features that enter late, or stay at 0 for a long time, add little that the others do not already have. For correlated features, the one that enters first is not necessarily "better", just chosen first.

In [ ]:
Cs = np.logspace(-4, 0, 13)
path = []
for c in Cs:
    m = LogisticRegression(penalty="l1", C=c, solver="liblinear", max_iter=2000).fit(Ztr, tr_s["y"])
    path.append(pd.Series(m.coef_[0], index=lin_cols))
path = pd.DataFrame(path, index=Cs)
by_feature = path.abs().T.groupby(feature_of).sum().T

entry = by_feature.gt(1e-6).apply(lambda s: s.idxmax() if s.any() else np.nan)
order = (pd.DataFrame({"first non-zero at C": entry, "weight at C = 1": by_feature.iloc[-1].round(3)})
           .sort_values("first non-zero at C"))
order.insert(0, "rank", range(1, len(order) + 1))

fig, ax = plt.subplots(figsize=(9, 5))
for f in order.index:
    ax.plot(Cs, by_feature[f], marker="o", markersize=3, label=f)
ax.set_xscale("log")
ax.set_xlabel("C (inverse penalty strength; right = weaker penalty)")
ax.set_ylabel("sum of |weights| per feature")
ax.set_title("Lasso path: order in which the features enter")
ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()
order

**Conclusion.**
- **Lasso picks exact name, rarest shared word and token coverage first,** then the other similarities and the first/last name fields.
- **Script and phonetic match come last,** so they add little linear information.

**Next step:** check whether a linear model is good enough (next table).

### Table: how good are the linear models at the operating point?

**What it shows:** false-alarm rate (share of negative test pairs with an alert, in %) at 95–99.5% recall for Ridge, a Lasso with a moderate penalty, and gradient boosting (set B), all trained on the same 100,000 pairs and tested on the same test pairs.

**How to read it:** if the linear models are clearly worse, the features act partly through interactions that only the tree model can use, which is one more reason not to read importance from the linear weights alone.

In [ ]:
lasso = LogisticRegression(penalty="l1", C=0.01, solver="liblinear", max_iter=2000).fit(Ztr, tr_s["y"])
Xtr_b, Xte_b = design(tr_s, ALL), design(te0, ALL)
Xte_b = Xte_b.reindex(columns=Xtr_b.columns, fill_value=0)
gbm = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, random_state=C.RANDOM_SEED).fit(Xtr_b, tr_s["y"])
yt = te0["y"].to_numpy()
models = {"Ridge (L2, C=1)": ridge.predict_proba(Zte)[:, 1],
          f"Lasso (L1, C=0.01; {int((lasso.coef_ != 0).sum())} of {len(lin_cols)} columns non-zero)": lasso.predict_proba(Zte)[:, 1],
          "Gradient boosting (set B)": gbm.predict_proba(Xte_b)[:, 1]}
pd.DataFrame({name: {f"false-alarm % @ {r:.1%}": round(100 * false_alarms_at(yt, s, r)[1] / (yt == 0).sum(), 2) for r in RECALLS}
              | {"AUC": round(roc_auc_score(yt, s), 4)} for name, s in models.items()}).T

**Conclusion.**
- **The linear models need about 70% more false alarms than gradient boosting** at 99% recall: Ridge 5.24%, Lasso 5.89%, boosting 3.13%.
- **At 99.5% recall they need more than twice as many** (9.3% vs 4.3%), although the AUC is almost equal. The features work partly through interactions.

**Next step:** gradient boosting as main model; logistic regression as baseline (notebook 07).

## 14 · Feature selection: do the weakest features cause overfitting?

Sections 11–13 compared whole feature groups (8 vs 15). Here we test single features. Candidates for removal are the three weakest in Ridge and Lasso (section 13):

- **n_script** (writing system of the two names: same, different, mixed)
- **n_initials** (does one name use initials that match the other)
- **n_numbers** (numbers in the names: match, conflict, none)

Four checks:

1. **Train/test gap** for the gradient boosting model with all 15 and with 12 features (without the three). A large gap would be the direct sign of overfitting.
2. **Leave one feature out**, chosen on the **training data only**: each training set of the entity split is split again by entities (inner validation). The outer test set is not used for the choice, otherwise the selection would be fitted to the test set.
3. **12 vs 15 on the outer test set**, paired over the 5 entity splits.
4. **Subgroups:** how often does each of the three features take a non-default value, and does it help in exactly those pairs?

Decision rule agreed beforehand: a feature is dropped if the full model is not better in at least 4 of 5 splits **and** it does not help in its subgroup.

In [ ]:
WEAK3 = ["n_script", "n_initials", "n_numbers"]   # weakest in Ridge and Lasso (section 13)
ALL12 = [c for c in ALL if c not in WEAK3]

def fit_score(tr, te, cols):
    """Fit HGB on tr; return scores on tr and on te."""
    Xtr, Xte = design(tr, cols), design(te, cols)
    Xte = Xte.reindex(columns=Xtr.columns, fill_value=0)
    m = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, random_state=C.RANDOM_SEED)
    m.fit(Xtr, tr["y"])
    return m.predict_proba(Xtr)[:, 1], m.predict_proba(Xte)[:, 1]

def fa_rate(y, s, recall):
    """False alarms at the given recall, as share of negatives."""
    return false_alarms_at(y, s, recall)[1] / (y == 0).sum()

t0 = time.time()
gap_rows, outer_scores = [], {}
for rep in range(REPS):
    tr, te = entity_split(full_data, Q, C.RANDOM_SEED + rep)
    for name, cols in {"all 15": ALL, "12 (without weak 3)": ALL12}.items():
        s_tr, s_te = fit_score(tr, te, cols)
        if rep == 0:
            outer_scores[name] = s_te
        for part, df, s in [("train", tr, s_tr), ("test", te, s_te)]:
            y = df["y"].to_numpy()
            gap_rows.append({"rep": rep, "features": name, "part": part,
                             **{f"FA at {r:.1%}": fa_rate(y, s, r) for r in RECALLS}})
gap = pd.DataFrame(gap_rows)
print(f"{time.time() - t0:.0f}s")

### Table: train vs test (overfitting check)

**What it shows:** false alarms (share of negative pairs with an alarm) at each recall, on the training pairs and on the test pairs, mean over the 5 entity splits. Last row block: test minus train in percentage points.

**How to read it:** if the model memorised the training data, the training values would be much lower than the test values. A gap that is similar for 12 and 15 features means the three extra features add no overfitting of their own.

In [ ]:
fa_cols = [c for c in gap.columns if c.startswith("FA at")]
g = gap.groupby(["features", "part"])[fa_cols].mean()
diff = (g.xs("test", level="part") - g.xs("train", level="part")) * 100
display((g * 100).round(2).rename(columns=lambda c: c + " (%)"))
display(diff.round(2).rename(columns=lambda c: c + ": test − train (pp)"))

**Conclusion.** Training and test false alarms are close (gap 0.18 pp at 99% recall for all 15 features, 0.13 pp for 12), so there is no sign of overfitting, and the three weak features do not add to it.

**Next step:** test single features (next table).

In [ ]:
# leave one feature out, evaluated on an inner validation split of the TRAINING data only
t0 = time.time()
variants = {"(none)": ALL, **{f: [c for c in ALL if c != f] for f in ALL}, "weak 3 together": ALL12}
loo_rows = []
for rep in range(REPS):
    tr, _ = entity_split(full_data, Q, C.RANDOM_SEED + rep)
    itr, iva = entity_split(tr, Q, C.RANDOM_SEED + 100 + rep)
    y = iva["y"].to_numpy()
    for drop, cols in variants.items():
        _, s = fit_score(itr, iva, cols)
        loo_rows.append({"rep": rep, "dropped": drop, **{f"FA at {r:.1%}": fa_rate(y, s, r) for r in RECALLS}})
loo = pd.DataFrame(loo_rows)
print(f"{len(variants)} variants x {REPS} splits in {time.time() - t0:.0f}s; "
      f"inner train {len(itr):,}, inner validation {len(iva):,} pairs (last split)")

### Table: leave one feature out (inner validation, training data only)

**What it shows:** for each feature, how much the false-alarm rate rises (in percentage points) when that single feature is removed, compared with the model with all 15. Mean over the 5 splits. **"full better"** counts in how many of the 5 splits the full model has fewer false alarms at 99% recall. The last row removes the three weak features together.

**How to read it:** positive values mean the feature helps. A feature with a change close to 0 and "full better" in 3 or fewer splits is within the noise and a candidate for removal. Sorted by the change at 99% recall.

In [ ]:
base = loo[loo["dropped"] == "(none)"].set_index("rep")[fa_cols]
rows = []
for drop, grp in loo[loo["dropped"] != "(none)"].groupby("dropped"):
    d = (grp.set_index("rep")[fa_cols] - base) * 100
    rows.append({"dropped": drop, **{c + ": change (pp)": d[c].mean() for c in fa_cols},
                 "full better (of 5) at 99%": int((d["FA at 99.0%"] > 0).sum())})
loo_tab = pd.DataFrame(rows).set_index("dropped").sort_values("FA at 99.0%: change (pp)", ascending=False)
print(f"reference, all 15 (inner validation): FA at 99% = {base['FA at 99.0%'].mean():.2%}")
loo_tab.round(2)

**Conclusion.**
- **The full-name frequency matters most:** removing it costs 1.22 pp at 99% recall, then core similarity (0.57) and word frequency (0.25).
- **Exact name, token coverage, token-set similarity and phonetic match are within noise** on their own, because the other similarity scores carry the same information.

**Next step:** keep all 15 (the weak three help, next table). Removing the redundant similarity scores as a group would need its own test.

### Table: 12 vs 15 features on the outer test set

**What it shows:** false alarms at each recall for the model with all 15 and with 12 features (without n_script, n_initials, n_numbers), on the outer test sets of the 5 entity splits. **"15 better"** counts the splits where all 15 have fewer false alarms.

**How to read it:** this is the confirmation on data that played no part in the choice. If 12 features are as good as 15, the three can go.

In [ ]:
t = gap[gap["part"] == "test"].pivot(index="rep", columns="features", values=fa_cols)
rows = []
for c in fa_cols:
    a, b = t[(c, "all 15")], t[(c, "12 (without weak 3)")]
    rows.append({"recall": c.replace("FA at ", ""), "all 15 (%)": a.mean() * 100, "12 (%)": b.mean() * 100,
                 "12 − 15 (pp)": (b - a).mean() * 100, "15 better (of 5)": int((b > a).sum())})
pd.DataFrame(rows).set_index("recall").round(2)

**Conclusion.** On the outer test sets all 15 beat 12 features in 5 of 5 splits at 98–99.5% recall (2.92% vs 3.13% at 99%).

**Next step:** keep all 15 features.

### Table: subgroups where the weak features take a non-default value

**What it shows:** for each of the three features, the test pairs (outer split 0) where the feature is **not** at its most common value. Within this subgroup: number of pairs and share of positives, then false alarms (negatives with an alarm) and misses (positives without an alarm) at the 99%-recall threshold, for the full model and for the model without this one feature.

**How to read it:** a feature can be weak on average but matter for a small group (for example names in different writing systems). If the counts with and without the feature are about the same in its own subgroup, it does not help there either.

In [ ]:
tr0, te0 = entity_split(full_data, Q, C.RANDOM_SEED)
y0 = te0["y"].to_numpy()

def alarms(s):
    thr, _ = false_alarms_at(y0, s, 0.99)
    return s >= thr

al_full = alarms(outer_scores["all 15"])
rows = []
for f in WEAK3:
    _, s_wo = fit_score(tr0, te0, [c for c in ALL if c != f])
    al_wo = alarms(s_wo)
    v = te0[f].astype(object).where(te0[f].notna(), "(missing)").astype(str)
    mode = v.mode()[0]
    sub = (v != mode).to_numpy()
    rows.append({"feature": f, "default value": mode, "pairs in subgroup": int(sub.sum()),
                 "share of test pairs": sub.mean(), "share positive": y0[sub].mean() if sub.any() else np.nan,
                 "false alarms, all 15": int((al_full & sub & (y0 == 0)).sum()),
                 "false alarms, without it": int((al_wo & sub & (y0 == 0)).sum()),
                 "misses, all 15": int((~al_full & sub & (y0 == 1)).sum()),
                 "misses, without it": int((~al_wo & sub & (y0 == 1)).sum())})
pd.DataFrame(rows).set_index("feature").round(3)

**Conclusion.**
- **Script and initials matter in their subgroups:** `n_script` differs from "same" in 22% of the test pairs, and there the full model misses fewer true matches (84 vs 94). `n_initials` shows the same in 13% of the pairs (74 vs 78).
- **`n_numbers` is rare** (2.4%) and neutral.

**Next step:** none; decision documented in `docs/decisions.md`.

### Findings: section 14

- **No sign of overfitting.** The gap between training and test is small: at 99% recall about 0.2 percentage points (pp) for both 15 and 12 features, at 99.5% about 0.6 pp. The three weak features add almost nothing to the gap.
- **The weak three still help.** On the outer test sets, all 15 beat 12 in 5 of 5 splits at 98, 99 and 99.5% recall: 2.92% vs 3.13% false alarms at 99%, 4.40% vs 4.75% at 99.5%. On the inner validation, removing n_initials or n_script makes it worse in 5/5 or 4/5 splits.
- **Subgroups:** n_script differs from "same" in 22% of test pairs and n_initials in 13%. There the full model has fewer misses (84 vs 94, 74 vs 78). n_numbers is rare (2.4% of pairs) and neutral there.
- **Decision (rule from above): keep all 15.** None of the three meets the removal rule. n_numbers is the closest case (4/5 splits, neutral in its subgroup).
- **Side finding:** single-feature removal is a different ranking from Ridge/Lasso. In the noise are n_exact_name (full model better in only 1 of 5 splits), n_token_coverage, n_sim_set and n_phonetic_match (3 of 5). They are not useless but redundant: the other similarity measures carry the same information. Removing them as a group would need its own test.

## Open questions / decisions

- Which numeric similarity to keep? Probably 1–2 of `n_sim_sort`, `n_sim_set`, `n_jaro_winkler` (see correlations).
- Do `n_token_coverage` and `n_phonetic_match` add anything beyond the whole-name similarity?
- Are the structural features (`n_lastname_status`, `n_numbers`, `n_initials`) useful at least for some pairs, even if rare?
- For companies: is `n_sim_core` better than `n_sim_sort`, and is `n_legal_form = conflict` a strong negative signal? Is the legal-form list good enough?
- Encoding for the final feature file: no `NaN` (missing names → `name_missing = 1`, numeric values 0).
- Re-run on the cleaned pairs and on all rows before deciding for good.